In [1]:
%load_ext autoreload
%autoreload 2

In [ ]:
import torch
from datasets import load_dataset
from torchvision.transforms import v2
import dotenv
import os

import ssl
from torch.utils.data import Dataset, random_split
from torchvision.datasets import MNIST as TorchMNIST
from torchvision.transforms import v2

import utils.cifar10_classification

In [3]:
dotenv.load_dotenv()
#print(os.getenv("HF_TOKEN")) #HF_TOKEN prescence in envvars is automatically detected by huggingface functions

True

#### Preprocessing
1. `v2.ToImage()` converts PIL image objects (that live on RAM) into `torch.tensor()` without dividing the pixels values by `255` like the deprecated `v2.ToTensor()`
2. `v2.GrayScale()` changes the number of channels of the input to `num_output_channels` which is redundant for MNIST here
3. `v2.Resize((32,32))` changes the spatial dimensions (height and width) to $32 \times 32$ while keeping proportions
4. `v2.RandomCrop(32, padding=4)` crops random part of the images to enable better generalization of the model so it can recognize the same objects even when they  
\\ are shifted across the image. `padding=4` adds 4 pixes to the spatial dimensions making the images $40\times 40$ and then crop 32 by 32 square image (which `torchvision` assumes is square of 32 by 32 which is equivalent to `(32, 32)` ) 
5. `v2.RandomHorizontalFlip(p=0.5)` horizontally flop images with change of 50%
6. `v2.ToDtype(torch.float32, scale=True)` scale all pixels' values into `float32` (which uses 32 bits to represent decimals) and then `scale=True` divides all by \\ `255` before conversion
7. `v2.Normalize(mean=[0.1307, 0.1307, 0.1307], std=[0.3081, 0.3081, 0.3081])` subtract all pixels in all 3 dimensions by the global mean of the dataset and divide \\ by the std (z-score normalization) 


In [ ]:

ssl._create_default_https_context = ssl._create_unverified_context

train_transform = v2.Compose([
    v2.ToImage(),
    v2.Grayscale(num_output_channels=3),  # Converts 1-channel MNIST to 3-channel pseudo-RGB
    v2.Resize((32, 32)),                  # Upscales from 28x28 to your original 32x32 size
    v2.RandomCrop(32, padding=4), 
    v2.RandomHorizontalFlip(p=0.5), 
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(mean=[0.1307, 0.1307, 0.1307], std=[0.3081, 0.3081, 0.3081]) 
])

eval_transform = v2.Compose([
    v2.ToImage(),
    v2.Grayscale(num_output_channels=3),
    v2.Resize((32, 32)),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(mean=[0.1307, 0.1307, 0.1307], std=[0.3081, 0.3081, 0.3081])
])

# 3. Download and load via Torchvision (Fast ~11MB download total)
full_train_raw = TorchMNIST(root='./data', train=True, download=True)
full_test_raw = TorchMNIST(root='./data', train=False, download=True)

# 4. Slice the raw data to match your percentages (5% train, 2% test)
train_size = int(0.05 * len(full_train_raw))   # 5% of 60,000 = 3,000
test_size = int(0.02 * len(full_test_raw))     # 2% of 10,000 = 200

train_subset, _ = random_split(full_train_raw, [train_size, len(full_train_raw) - train_size])
test_subset, _ = random_split(full_test_raw, [test_size, len(full_test_raw) - test_size])

Training dataset size: 3000
Validation dataset size: 100
Testing dataset size: 100


#### Dataset
the dataset class may inherit (although it is not necessary) from `torch.utils.data.Dataset` class and must define
`__init__()` for initializing and required info like metadata
`__len__()` that returns the number of samples
`__getitem__()` which returns the (input, output) pair in supervised learning and a sample in unsupervised learning

In [ ]:

class MNISTWrapper(Dataset):
    def __init__(self, subset, transform):
        self.subset = subset
        self.transform = transform

    def __len__(self):
        return len(self.subset)

    def __getitem__(self, idx):
        img, label = self.subset[idx] 
        img = self.transform(img)
        return img, label

dataset_train = MNISTWrapper(train_subset, train_transform)
print(f"Training dataset size: {len(dataset_train)}") # Outputs 3000

val_size = len(test_subset) // 2
test_slice_1, test_slice_2 = random_split(test_subset, [val_size, len(test_subset) - val_size])

dataset_val = MNISTWrapper(test_slice_1, eval_transform)
dataset_test = MNISTWrapper(test_slice_2, eval_transform)

print(f"Validation dataset size: {len(dataset_val)}") # Outputs 100
print(f"Testing dataset size: {len(dataset_test)}")   # Outputs 100


In [5]:
train_loader = torch.utils.data.DataLoader(dataset_train, batch_size=64, shuffle=True)
test_loader = torch.utils.data.DataLoader(dataset_test, batch_size=64, shuffle=True)
val_loader = torch.utils.data.DataLoader(dataset_val, batch_size=64, shuffle=True)

##### BasicNet
Consists of Conv2D -> ReLU -> Linear

If $x$ is a single CIFAR10 samples such $x \in \mathbb{R}^{}$

In [ ]:
import architectures.BasicNet

BasicNet = architectures.BasicNet.BasicNet()
bn = BasicNet()
utils.cifar10_classification.train(bn,train_loader, val_loader)
utils.cifar10_classification.test(bn,test_loader)